In [1]:
import numpy as np
import tifffile
import cv2
import os

def analyze_and_colorize_mask(mask_path: str):
    """
    Загружает TIFF маску, выводит статистику по классам 
    и сохраняет цветную RGB-версию.
    """
    if not os.path.exists(mask_path):
        print(f"Ошибка: Файл {mask_path} не найден.")
        return

    # 1. Загрузка оригинальной маски (без потери битности)
    mask = tifffile.imread(mask_path)
    print(f"Исходный размер маски: {mask.shape}")
    
    # 2. Анализ классов
    # return_inverse=True автоматически создаст новую маску, 
    # где классы идут строго по порядку: 0, 1, 2, 3... 
    # Это защищает от сбоев, если в маске значения типа 0 и 255.
    unique_classes, safe_indexed_mask = np.unique(mask, return_inverse=True)
    safe_indexed_mask = safe_indexed_mask.reshape(mask.shape)
    
    num_classes = len(unique_classes)
    print(f"Найдено уникальных классов: {num_classes}")
    print(f"Физические значения пикселей в файле: {unique_classes}")
    
    for i, cls in enumerate(unique_classes):
        pixel_count = np.sum(mask == cls)
        print(f" - Класс {cls}: {pixel_count} пикселей")

    # 3. Создание красивой палитры (RGB)
    # Задаем жесткие контрастные цвета для первых классов
    predefined_colors = [
        [0, 0, 0],       # Фон: Черный
        [255, 0, 0],     # Класс 1: Красный
        [0, 255, 0],     # Класс 2: Зеленый
        [0, 0, 255],     # Класс 3: Синий
        [255, 255, 0],   # Класс 4: Желтый
        [255, 0, 255],   # Класс 5: Маджента
        [0, 255, 255]    # Класс 6: Циан
    ]
    
    palette = np.zeros((num_classes, 3), dtype=np.uint8)
    
    for i in range(num_classes):
        if i < len(predefined_colors):
            palette[i] = predefined_colors[i]
        else:
            # Если классов больше 7, генерируем случайный яркий цвет
            palette[i] = np.random.randint(50, 255, size=3)

    # 4. Мгновенная векторизованная раскраска
    colored_mask_rgb = palette[safe_indexed_mask]

    # 5. Сохранение результата
    # OpenCV исторически работает в формате BGR, поэтому перед сохранением меняем каналы
    colored_mask_bgr = cv2.cvtColor(colored_mask_rgb, cv2.COLOR_RGB2BGR)
    
    output_path = mask_path.replace(".tif", "_colored.png")
    cv2.imwrite(output_path, colored_mask_bgr)
    print(f"\nУспех! Цветная версия сохранена как: {output_path}")

# ==========================================
# ЗАПУСК
# ==========================================
# Замените на путь к вашему реальному файлу
my_mask_path = r"C:\Users\semen\SEprojects\archive\2DeteCT_slices3001-4000_RecSeg\slice03074\mode2\segmentation.tif"

analyze_and_colorize_mask(my_mask_path)

Исходный размер маски: (1024, 1024)
Найдено уникальных классов: 4
Физические значения пикселей в файле: [1 2 3 4]
 - Класс 1: 396597 пикселей
 - Класс 2: 47708 пикселей
 - Класс 3: 498799 пикселей
 - Класс 4: 105472 пикселей

Успех! Цветная версия сохранена как: C:\Users\semen\SEprojects\archive\2DeteCT_slices3001-4000_RecSeg\slice03074\mode2\segmentation_colored.png


In [7]:
import numpy as np
import tifffile
import nibabel as nib
from pathlib import Path

def build_balanced_ultra_light_dataset_fixed(root_dir: str, out_dir: str):
    root_path = Path(root_dir)
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    rec_slices = []
    seg_slices = []

    # 1. Находим все папки со срезами и сортируем их
    all_slice_dirs = sorted(root_path.glob("[sS]lice*"))
    target_slice_dirs = all_slice_dirs[::2]
    print(f"Всего папок в архиве: {len(all_slice_dirs)}. В обработку идет: {len(target_slice_dirs)}")

    print("\n2. Чтение файлов со сжатием геометрии (1024x1024 -> 512x512)...")
    for s_dir in target_slice_dirs:
        mode2_dir = s_dir / "mode2"

        if not mode2_dir.exists():
            continue

        rec_files = list(mode2_dir.glob("*reconstruction*.tif*"))
        seg_files = list(mode2_dir.glob("*segmentation*.tif*"))

        if rec_files and seg_files:
            rec_slices.append(tifffile.imread(rec_files[0])[::2, ::2])
            seg_slices.append(tifffile.imread(seg_files[0])[::2, ::2])

    if not rec_slices:
        print("Ошибка: Не удалось загрузить файлы.")
        return

    # Сборка в 3D
    vol_rec = np.transpose(np.stack(rec_slices), (2, 1, 0))
    vol_seg = np.transpose(np.stack(seg_slices), (2, 1, 0))
    
    print(f"\nНовая геометрия 3D-тензора: {vol_rec.shape} (X, Y, Z)")

    affine = np.eye(4)
    affine[0, 0] = 2.0  
    affine[1, 1] = 2.0  
    affine[2, 2] = 2.0  

    print("\n3. Нормализация и сохранение реконструкции в uint16...")
    v_min, v_max = vol_rec.min(), vol_rec.max()
    if v_max - v_min == 0:
        vol_rec_uint16 = vol_rec.astype(np.uint16)
    else:
        vol_rec_normalized = ((vol_rec - v_min) / (v_max - v_min)) * 65535.0
        vol_rec_uint16 = vol_rec_normalized.astype(np.uint16)
    
    nii_rec = nib.Nifti1Image(vol_rec_uint16, affine)
    nii_rec.set_data_dtype(np.uint16)
    nib.save(nii_rec, out_path / "3D_reconstruction_512_uint16.nii.gz")
    print(" - 3D_reconstruction_512_uint16.nii.gz успешно сохранен.")
    
    del vol_rec, vol_rec_uint16, nii_rec
    if 'vol_rec_normalized' in locals(): del vol_rec_normalized


    # ==========================================
    # ИСПРАВЛЕНИЕ СДВИГА МАСОК
    # ==========================================
    print("\n4. Корректировка индексов масок (Исправление сдвига)...")
    unique_vals = np.unique(vol_seg)
    print(f"Обнаружены сырые значения классов в TIFF: {unique_vals}")

    # Создаем новый чистый массив
    vol_seg_fixed = np.zeros_like(vol_seg, dtype=np.uint8)

    # ЖЕСТКИЙ МАППИНГ: Убираем съезд на 1
    # Если в массиве есть другие значения, они просто останутся нулями (фоном)
    vol_seg_fixed[vol_seg == 1] = 0  # Значение 1 становится 0 (Фон)
    vol_seg_fixed[vol_seg == 2] = 1  # Значение 2 становится 1 (Банка)
    vol_seg_fixed[vol_seg == 3] = 2  # Значение 3 становится 2 (Наполнитель)
    vol_seg_fixed[vol_seg == 4] = 3  # Значение 4 становится 3 (Твердые предметы)

    # Очищаем старый кривой массив из памяти
    del vol_seg

    # ==========================================

    classes = {
        0: "mask_0_bg_512.nii.gz",
        1: "mask_1_can_512.nii.gz",
        2: "mask_2_filler_512.nii.gz",
        3: "mask_3_hard_subjects_512.nii.gz"
    }

    print("\n5. Экспорт масок классов (в формате uint8 0/1)...")
    for class_idx, filename in classes.items():
        # Обратите внимание: теперь мы берем данные из ИСПРАВЛЕННОГО массива (vol_seg_fixed)
        mask_uint8 = (vol_seg_fixed == class_idx).astype(np.uint8)
        
        nii_mask = nib.Nifti1Image(mask_uint8, affine)
        nii_mask.set_data_dtype(np.uint8) 
        
        nib.save(nii_mask, out_path / filename)
        print(f" - {filename} успешно сохранен.")
        
        del mask_uint8, nii_mask

    print("\n6. Сохранение общей индексной маски всех классов...")
    nii_all = nib.Nifti1Image(vol_seg_fixed, affine)
    nii_all.set_data_dtype(np.uint8)
    nib.save(nii_all, out_path / "mask_ALL_classes_512.nii.gz")
    print(" - mask_ALL_classes_512.nii.gz успешно сохранен.")
    
    print("\n[УСПЕХ] Сборка датасета завершена! Твердые предметы восстановлены.")

# ==========================================
# ИНИЦИАЛИЗАЦИЯ ПУТЕЙ И ЗАПУСК
# ==========================================
root_directory = r"C:\Users\semen\SEprojects\archive\2DeteCT_slices3001-4000_RecSeg"
output_directory = r"C:\Users\semen\SEprojects\archive\3D_NIFTI_Models_512_Z2"

build_balanced_ultra_light_dataset_fixed(root_directory, output_directory)

Всего папок в архиве: 1000. В обработку идет: 500

2. Чтение файлов со сжатием геометрии (1024x1024 -> 512x512)...

Новая геометрия 3D-тензора: (512, 512, 500) (X, Y, Z)

3. Нормализация и сохранение реконструкции в uint16...
 - 3D_reconstruction_512_uint16.nii.gz успешно сохранен.

4. Корректировка индексов масок (Исправление сдвига)...
Обнаружены сырые значения классов в TIFF: [1 2 3 4]

5. Экспорт масок классов (в формате uint8 0/1)...
 - mask_0_bg_512.nii.gz успешно сохранен.
 - mask_1_can_512.nii.gz успешно сохранен.
 - mask_2_filler_512.nii.gz успешно сохранен.
 - mask_3_hard_subjects_512.nii.gz успешно сохранен.

6. Сохранение общей индексной маски всех классов...
 - mask_ALL_classes_512.nii.gz успешно сохранен.

[УСПЕХ] Сборка датасета завершена! Твердые предметы восстановлены.


In [4]:
import numpy as np
import tifffile
import nibabel as nib
from pathlib import Path
from scipy.ndimage import binary_closing, binary_opening

def build_industrial_dataset_pipeline(root_dir: str, out_dir: str):
    root_path = Path(root_dir)
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    rec_slices = []
    seg_slices = []

    # ==========================================
    # ШАГ 1: Поиск файлов и децимация по оси Z
    # ==========================================
    all_slice_dirs = sorted(root_path.glob("[sS]lice*"))
    
    # Берем каждый второй срез для оптимизации памяти и скорости
    target_slice_dirs = all_slice_dirs[::2]
    print(f"[*] Всего папок в архиве: {len(all_slice_dirs)}. В обработку идет: {len(target_slice_dirs)}")

    # ==========================================
    # ШАГ 2: Чтение данных и субдискретизация (Сжатие X, Y)
    # ==========================================
    print("\n[*] Чтение файлов со сжатием геометрии (1024x1024 -> 512x512)...")
    for s_dir in target_slice_dirs:
        mode2_dir = s_dir / "mode2"
        if not mode2_dir.exists():
            continue

        rec_files = list(mode2_dir.glob("*reconstruction*.tif*"))
        seg_files = list(mode2_dir.glob("*segmentation*.tif*"))

        if rec_files and seg_files:
            # Считываем и берем каждый второй пиксель по строкам и столбцам [::2, ::2]
            rec_slices.append(tifffile.imread(rec_files[0])[::2, ::2])
            seg_slices.append(tifffile.imread(seg_files[0])[::2, ::2])

    if not rec_slices:
        print("[!] Ошибка: Не удалось загрузить файлы. Проверьте пути.")
        return

    # Собираем 3D-тензор и транспонируем в медицинский стандарт NIfTI (X, Y, Z)
    vol_rec = np.transpose(np.stack(rec_slices), (2, 1, 0))
    vol_seg = np.transpose(np.stack(seg_slices), (2, 1, 0))
    print(f"[*] Новая геометрия 3D-тензора: {vol_rec.shape} (X, Y, Z)")

    # Аффинная матрица: физический размер вокселя увеличен в 2 раза по всем осям
    affine = np.eye(4)
    affine[0, 0] = 2.0  
    affine[1, 1] = 2.0  
    affine[2, 2] = 2.0  

    # ==========================================
    # ШАГ 3: Нормализация Реконструкции
    # ==========================================
    print("\n[*] Нормализация снимков и сохранение в формат uint16...")
    v_min, v_max = vol_rec.min(), vol_rec.max()
    if v_max - v_min == 0:
        vol_rec_uint16 = vol_rec.astype(np.uint16)
    else:
        # Растягиваем гистограмму плотностей на полный диапазон 16-битных целых чисел
        vol_rec_uint16 = (((vol_rec - v_min) / (v_max - v_min)) * 65535.0).astype(np.uint16)
    
    nii_rec = nib.Nifti1Image(vol_rec_uint16, affine)
    nii_rec.set_data_dtype(np.uint16)
    nib.save(nii_rec, out_path / "3D_reconstruction_512_uint16.nii.gz")
    print("    -> 3D_reconstruction_512_uint16.nii.gz успешно сохранен.")
    
    # Очищаем память от тяжелых снимков, оставляем только маски
    del vol_rec, vol_rec_uint16, nii_rec

    # ==========================================
    # ШАГ 4: Жесткий маппинг классов (Исправление сдвига TIFF)
    # ==========================================
    print("\n[*] Корректировка индексов масок для nnU-Net...")
    vol_seg_fixed = np.zeros_like(vol_seg, dtype=np.uint8)
    
    # Сырые классы из TIFF переводим в строгий стандарт [0, 1, 2, 3]
    vol_seg_fixed[vol_seg == 1] = 0  # Фон
    vol_seg_fixed[vol_seg == 2] = 1  # Банка
    vol_seg_fixed[vol_seg == 3] = 2  # Наполнитель
    vol_seg_fixed[vol_seg == 4] = 3  # Твердые предметы
    del vol_seg

    # ==========================================
    # ШАГ 5: Агрессивная пространственная морфология (Для макрообъектов)
    # ==========================================
    print("\n[*] Запуск модуля морфологической оптимизации макрообъектов...")
    mask_hard = (vol_seg_fixed == 3)
    
    # 5.1 ОТКРЫТИЕ (Удаление лучевых артефактов и мелкой пыли)
    # Ядро 3x3x3, 3 итерации. Сотрет артефакты толщиной до 6 вокселей.
    opened_mask = binary_opening(mask_hard, structure=np.ones((3, 3, 3)), iterations=2)
    
    # Вычисляем удаленный мусор и мгновенно затираем его фоном (0) в главном массиве
    garbage_pixels = mask_hard & (~opened_mask)
    vol_seg_fixed[garbage_pixels] = 0
    print(f"    -> Удалено вокселей шумовой пыли: {np.sum(garbage_pixels)}")
    
    # 5.2 ЗАКРЫТИЕ (Монолитное склеивание "нарезанных" слоев)
    # Большое ядро 5x5x5, 2 итерации. Перепрыгнет пустоты Z-слайсинга (до 8-10 вокселей).
    closed_mask = binary_closing(opened_mask, structure=np.ones((4, 4, 4)), iterations=2)
    
    # 5.3 БЕЗОПАСНОЕ ВНЕДРЕНИЕ С ЗАЩИТОЙ КОЛЛИЗИЙ
    # Новый клей пишем только туда, где сейчас пусто (0) или мягкий наполнитель (2).
    # Стенки пластиковой банки (1) остаются защищенными от перезаписи!
    safe_glue = (closed_mask & (~opened_mask)) & ((vol_seg_fixed == 0) | (vol_seg_fixed == 2))
    vol_seg_fixed[safe_glue] = 3
    print(f"    -> Синтезировано вокселей связующего клея: {np.sum(safe_glue)}")

    # ==========================================
    # ШАГ 6: Экспорт изолированных масок
    # ==========================================
    print("\n[*] Экспорт финальных масок в NIfTI...")
    classes = {
        0: "mask_0_bg_512.nii.gz",
        1: "mask_1_can_512.nii.gz",
        2: "mask_2_filler_512.nii.gz",
        3: "mask_3_hard_subjects_512.nii.gz"
    }

    # Сохраняем каждую маску отдельно в формате 0/1 (uint8)
    for class_idx, filename in classes.items():
        nii_mask = nib.Nifti1Image((vol_seg_fixed == class_idx).astype(np.uint8), affine)
        nii_mask.set_data_dtype(np.uint8) 
        nib.save(nii_mask, out_path / filename)
        print(f"    -> {filename} сохранен.")

    # Сохраняем общий файл для обучения nnU-Net
    nii_all = nib.Nifti1Image(vol_seg_fixed, affine)
    nii_all.set_data_dtype(np.uint8)
    nib.save(nii_all, out_path / "mask_ALL_classes_512.nii.gz")
    print("    -> mask_ALL_classes_512.nii.gz сохранен.")
    
    print("\n[✔] Пайплайн успешно завершен!")

# ==========================================
# ТОЧКА ВХОДА (Задайте ваши пути)
# ==========================================
if __name__ == "__main__":
    root_directory = r"C:\Users\semen\SEprojects\archive\2DeteCT_slices3001-4000_RecSeg"
    output_directory = r"C:\Users\semen\SEprojects\archive\3D_NIFTI_Models_512_Z2_Final"

    build_industrial_dataset_pipeline(root_directory, output_directory)

[*] Всего папок в архиве: 1000. В обработку идет: 500

[*] Чтение файлов со сжатием геометрии (1024x1024 -> 512x512)...
[*] Новая геометрия 3D-тензора: (512, 512, 500) (X, Y, Z)

[*] Нормализация снимков и сохранение в формат uint16...
    -> 3D_reconstruction_512_uint16.nii.gz успешно сохранен.

[*] Корректировка индексов масок для nnU-Net...

[*] Запуск модуля морфологической оптимизации макрообъектов...
    -> Удалено вокселей шумовой пыли: 14703440
    -> Синтезировано вокселей связующего клея: 729838

[*] Экспорт финальных масок в NIfTI...
    -> mask_0_bg_512.nii.gz сохранен.
    -> mask_1_can_512.nii.gz сохранен.
    -> mask_2_filler_512.nii.gz сохранен.
    -> mask_3_hard_subjects_512.nii.gz сохранен.
    -> mask_ALL_classes_512.nii.gz сохранен.

[✔] Пайплайн успешно завершен!


In [1]:
import numpy as np
import tifffile
import nibabel as nib
from pathlib import Path
from scipy import ndimage

def build_full_res_cropped_dataset(root_dir: str, out_dir: str, start_slice: int = 200, end_slice: int = 400):
    """
    Собирает 3D-датасет из непрерывного блока срезов в оригинальном разрешении 1024x1024.
    
    :param start_slice: Индекс начального среза (включительно)
    :param end_slice: Индекс конечного среза (не включительно)
    """
    root_path = Path(root_dir)
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    rec_slices = []
    seg_slices = []

    # ==========================================
    # ШАГ 1: Поиск и срез (Cropping) по оси Z
    # ==========================================
    all_slice_dirs = sorted(root_path.glob("[sS]lice*"))
    
    # Защита от выхода за пределы архива
    total_slices = len(all_slice_dirs)
    actual_end = min(end_slice, total_slices)
    
    target_slice_dirs = all_slice_dirs[start_slice:actual_end]
    
    print(f"[*] Всего папок в архиве: {total_slices}.")
    print(f"[*] Вырезан блок Z: со среза {start_slice} по {actual_end} (Итого: {len(target_slice_dirs)} слоев).")

    # ==========================================
    # ШАГ 2: Чтение данных (БЕЗ СЖАТИЯ)
    # ==========================================
    print("\n[*] Чтение TIFF файлов в 100% разрешении (1024x1024)...")
    for s_dir in target_slice_dirs:
        mode2_dir = s_dir / "mode2"
        if not mode2_dir.exists():
            continue

        rec_files = list(mode2_dir.glob("*reconstruction*.tif*"))
        seg_files = list(mode2_dir.glob("*segmentation*.tif*"))

        if rec_files and seg_files:
            # Читаем массивы целиком, без субдискретизации [::2]
            rec_slices.append(tifffile.imread(rec_files[0]))
            seg_slices.append(tifffile.imread(seg_files[0]))

    if not rec_slices:
        print("[!] Ошибка: Не удалось загрузить файлы. Проверьте пути.")
        return

    # Собираем 3D-тензор (X, Y, Z)
    vol_rec = np.transpose(np.stack(rec_slices), (2, 1, 0))
    vol_seg = np.transpose(np.stack(seg_slices), (2, 1, 0))
    print(f"[*] Итоговая геометрия ROI-тензора: {vol_rec.shape} (X, Y, Z)")

    # Аффинная матрица: так как сжатия нет, шаг сетки возвращаем к 1.0
    ratio_z = 1.0 / 0.1138  # Учитывая, что мы берем каждый второй срез, шаг по Z увеличивается в 2 раза
    affine = np.eye(4)
    affine[0, 0] = 1.0  
    affine[1, 1] = 1.0  
    affine[2, 2] = 1.0 * ratio_z
    # ==========================================
    # ШАГ 3: Нормализация Реконструкции
    # ==========================================
    print("\n[*] Нормализация снимков (uint16)...")
    v_min, v_max = vol_rec.min(), vol_rec.max()
    if v_max - v_min == 0:
        vol_rec_uint16 = vol_rec.astype(np.uint16)
    else:
        vol_rec_uint16 = (((vol_rec - v_min) / (v_max - v_min)) * 65535.0).astype(np.uint16)
    
    nii_rec = nib.Nifti1Image(vol_rec_uint16, affine)
    nii_rec.set_data_dtype(np.uint16)
    nib.save(nii_rec, out_path / f"3D_reconstruction_Z{start_slice}-{actual_end}_uint16.nii.gz")
    del vol_rec, vol_rec_uint16, nii_rec

    # ==========================================
    # ШАГ 4: Корректировка классов (nnU-Net стандарт)
    # ==========================================
    print("\n[*] Исправление сдвига классов маски...")
    vol_seg_fixed = np.zeros_like(vol_seg, dtype=np.uint8)
    
    vol_seg_fixed[vol_seg == 1] = 0  # Фон
    vol_seg_fixed[vol_seg == 2] = 1  # Банка
    vol_seg_fixed[vol_seg == 3] = 2  # Наполнитель
    vol_seg_fixed[vol_seg == 4] = 3  # Твердые предметы
    del vol_seg
    # ==========================================

    # ==========================================
    # ШАГ 5: Экспорт масок
    # ==========================================
    print("\n[*] Экспорт NIfTI масок...")
    classes = {
        0: f"mask_0_bg_Z{start_slice}-{actual_end}.nii.gz",
        1: f"mask_1_can_Z{start_slice}-{actual_end}.nii.gz",
        2: f"mask_2_filler_Z{start_slice}-{actual_end}.nii.gz",
        3: f"mask_3_hard_subjects_Z{start_slice}-{actual_end}.nii.gz"
    }

    for class_idx, filename in classes.items():
        nii_mask = nib.Nifti1Image((vol_seg_fixed == class_idx).astype(np.uint8), affine)
        nii_mask.set_data_dtype(np.uint8) 
        nib.save(nii_mask, out_path / filename)

    nii_all = nib.Nifti1Image(vol_seg_fixed, affine)
    nii_all.set_data_dtype(np.uint8)
    nib.save(nii_all, out_path / f"mask_ALL_classes_Z{start_slice}-{actual_end}.nii.gz")
    
    print("\n[✔] Сборка датасета без потерь качества завершена!")

# ==========================================
# ЗАПУСК
# ==========================================
if __name__ == "__main__":
    root_directory = r"C:\Users\semen\SEprojects\archive\2DeteCT_slices3001-4000_RecSeg"
    output_directory = r"C:\Users\semen\SEprojects\archive\3D_NIFTI_Models_1024_Cropped"

    # Регулируйте эти параметры по необходимости
    build_full_res_cropped_dataset(
        root_dir=root_directory, 
        out_dir=output_directory, 
        start_slice=200, 
        end_slice=400
    )

[*] Всего папок в архиве: 1000.
[*] Вырезан блок Z: со среза 200 по 400 (Итого: 200 слоев).

[*] Чтение TIFF файлов в 100% разрешении (1024x1024)...
[*] Итоговая геометрия ROI-тензора: (1024, 1024, 200) (X, Y, Z)

[*] Нормализация снимков (uint16)...

[*] Исправление сдвига классов маски...

[*] Экспорт NIfTI масок...

[✔] Сборка датасета без потерь качества завершена!


In [1]:
import numpy as np
import tifffile
import nibabel as nib
from pathlib import Path
from scipy.ndimage import zoom

def build_nnunet_isotropic_dataset(
    root_dir: str, 
    out_dir: str, 
    case_name: str = "banka_001",
    start_slice: int = 300, 
    end_slice: int = 375,
    xy_spacing: float = 0.1138, # Шаг пикселя в плоскости XY (в мм)
    z_spacing: float = 1.0      # Шаг между срезами (в мм)
):
    """
    Собирает 3D-датасет из непрерывного блока срезов, делает его изотропным
    (кубический воксель) и сохраняет в формате, готовом для nnU-Net.
    """
    root_path = Path(root_dir)
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    rec_slices = []
    seg_slices = []

    # ==========================================
    # ШАГ 1: Поиск и срез (Cropping) по оси Z
    # ==========================================
    all_slice_dirs = sorted(root_path.glob("[sS]lice*"))
    actual_end = min(end_slice, len(all_slice_dirs))
    target_slice_dirs = all_slice_dirs[start_slice:actual_end]
    
    print(f"[*] Вырезан блок Z: со среза {start_slice} по {actual_end} (Итого: {len(target_slice_dirs)} слоев).")

    # ==========================================
    # ШАГ 2: Чтение TIFF файлов (X, Y, Z)
    # ==========================================
    print("[*] Чтение TIFF файлов в 100% разрешении (1024x1024)...")
    for s_dir in target_slice_dirs:
        mode2_dir = s_dir / "mode2"
        if not mode2_dir.exists():
            continue

        rec_files = list(mode2_dir.glob("*reconstruction*.tif*"))
        seg_files = list(mode2_dir.glob("*segmentation*.tif*"))

        if rec_files and seg_files:
            rec_slices.append(tifffile.imread(rec_files[0]))
            seg_slices.append(tifffile.imread(seg_files[0]))

    if not rec_slices:
        print("[!] Ошибка: Не удалось загрузить файлы.")
        return

    # nnU-Net ожидает геометрию (X, Y, Z)
    vol_rec = np.transpose(np.stack(rec_slices), (2, 1, 0))
    vol_seg = np.transpose(np.stack(seg_slices), (2, 1, 0))
    print(f"[*] Исходная геометрия: {vol_rec.shape}")

    # ==========================================
    # ШАГ 3: Изотропная Интерполяция (Превращаем в куб)
    # ==========================================
    # Рассчитываем, во сколько раз нужно растянуть ось Z
    z_zoom_factor = z_spacing / xy_spacing
    zoom_factors = (1.0, 1.0, z_zoom_factor)
    
    print(f"\n[*] Запуск изотропной интерполяции...")
    print(f"[*] Коэффициент растяжения оси Z: {z_zoom_factor:.3f}")
    
    # Для КТ-снимков используем сплайны (order=3)
    vol_rec_iso = zoom(vol_rec, zoom_factors, order=3, mode='nearest')
    del vol_rec # Чистим ОЗУ
    
    # Для масок ЖЕСТКО Nearest Neighbor (order=0), иначе появятся дробные классы
    vol_seg_iso = zoom(vol_seg, zoom_factors, order=0, mode='nearest')
    del vol_seg # Чистим ОЗУ
    
    print(f"[*] Новая (изотропная) геометрия: {vol_rec_iso.shape}")

    # ==========================================
    # ШАГ 4: Корректировка классов (nnU-Net стандарт)
    # ==========================================
    print("\n[*] Переназначение классов маски (0 - фон)...")
    vol_seg_fixed = np.zeros_like(vol_seg_iso, dtype=np.uint8)
    vol_seg_fixed[vol_seg_iso == 1] = 0  # Фон
    vol_seg_fixed[vol_seg_iso == 2] = 1  # Банка
    vol_seg_fixed[vol_seg_iso == 3] = 2  # Наполнитель
    vol_seg_fixed[vol_seg_iso == 4] = 3  # Твердые предметы
    del vol_seg_iso

    # ==========================================
    # ШАГ 5: Формирование метаданных и экспорт (nnU-Net Ready)
    # ==========================================
    print("[*] Экспорт NIfTI файлов...")
    
    # Создаем базовую аффинную матрицу. Разрешение теперь везде xy_spacing!
    affine = np.eye(4)
    target_spacing = (xy_spacing, xy_spacing, xy_spacing)

    # 5.1 Сохранение снимка (Image)
    # Форматируем под 0-65535 (uint16) для экономии места
    v_min, v_max = vol_rec_iso.min(), vol_rec_iso.max()
    vol_rec_uint16 = (((vol_rec_iso - v_min) / (v_max - v_min)) * 65535.0).astype(np.uint16)
    
    # Имя обязательно должно заканчиваться на _0000.nii.gz
    img_filename = out_path / f"{case_name}_0000.nii.gz"
    nii_rec = nib.Nifti1Image(vol_rec_uint16, affine)
    nii_rec.header.set_zooms(target_spacing)
    nib.save(nii_rec, img_filename)
    del vol_rec_iso, vol_rec_uint16

    # 5.2 Сохранение единой маски (Label)
    # Имя должно точно совпадать с case_name, без _0000
    lbl_filename = out_path / f"{case_name}.nii.gz"
    nii_lbl = nib.Nifti1Image(vol_seg_fixed, affine)
    nii_lbl.header.set_zooms(target_spacing)
    nib.save(nii_lbl, lbl_filename)
    
    print(f"\n[✔] Готово! Данные можно переносить в папки imagesTr и labelsTr.")
    print(f"    Снимок: {img_filename.name}")
    print(f"    Маска:  {lbl_filename.name}")

# ==========================================
# ЗАПУСК
# ==========================================
if __name__ == "__main__":
    root_directory = r"C:\Users\semen\SEprojects\archive\2DeteCT_slices3001-4000_RecSeg"
    output_directory = r"C:\Users\semen\SEprojects\archive\nnUNet_Dataset\Task001_Banka"

    build_nnunet_isotropic_dataset(
        root_dir=root_directory, 
        out_dir=output_directory, 
        case_name="banka_crop_200", # Уникальное имя этого куска
        start_slice=300, 
        end_slice=375,
        xy_spacing=0.1138,
        z_spacing=1.0 # Замените на реальный физический шаг томографа по Z, если он отличается
    )

[*] Вырезан блок Z: со среза 300 по 375 (Итого: 75 слоев).
[*] Чтение TIFF файлов в 100% разрешении (1024x1024)...
[*] Исходная геометрия: (1024, 1024, 75)

[*] Запуск изотропной интерполяции...
[*] Коэффициент растяжения оси Z: 8.787
[*] Новая (изотропная) геометрия: (1024, 1024, 659)

[*] Переназначение классов маски (0 - фон)...
[*] Экспорт NIfTI файлов...

[✔] Готово! Данные можно переносить в папки imagesTr и labelsTr.
    Снимок: banka_crop_200_0000.nii.gz
    Маска:  banka_crop_200.nii.gz


In [2]:
import numpy as np
import tifffile
import nibabel as nib
import json
import gc
from pathlib import Path
from scipy.ndimage import zoom

def process_and_save_chunk(
    root_path: Path, 
    out_images_dir: Path, 
    out_labels_dir: Path,
    start_slice: int, 
    num_slices: int, 
    case_name: str, 
    xy_spacing: float, 
    z_spacing: float
):
    """Извлекает, интерполирует и сохраняет один кусок данных, экономя ОЗУ."""
    print(f"\n[{case_name}] НАЧАЛО ОБРАБОТКИ (Срезы {start_slice} - {start_slice + num_slices})")
    
    all_slice_dirs = sorted(root_path.glob("[sS]lice*"))
    actual_end = min(start_slice + num_slices, len(all_slice_dirs))
    target_dirs = all_slice_dirs[start_slice:actual_end]
    
    rec_slices, seg_slices = [], []
    
    # 1. ЧТЕНИЕ ДАННЫХ
    print(f"[{case_name}] Загрузка {len(target_dirs)} TIFF файлов в ОЗУ...")
    for s_dir in target_dirs:
        mode2_dir = s_dir / "mode2"
        if not mode2_dir.exists(): continue
        
        rec_files = list(mode2_dir.glob("*reconstruction*.tif*"))
        seg_files = list(mode2_dir.glob("*segmentation*.tif*"))
        
        if rec_files and seg_files:
            rec_slices.append(tifffile.imread(rec_files[0]))
            seg_slices.append(tifffile.imread(seg_files[0]))

    # (X, Y, Z)
    vol_rec = np.transpose(np.stack(rec_slices), (2, 1, 0))
    vol_seg = np.transpose(np.stack(seg_slices), (2, 1, 0))
    
    # Очищаем списки из ОЗУ
    del rec_slices, seg_slices
    gc.collect()

    # 2. ИНТЕРПОЛЯЦИЯ ДО КУБА
    z_zoom_factor = z_spacing / xy_spacing
    zoom_factors = (1.0, 1.0, z_zoom_factor)
    
    print(f"[{case_name}] Интерполяция снимка (ось Z увеличится в {z_zoom_factor:.2f} раз)...")
    vol_rec_iso = zoom(vol_rec, zoom_factors, order=3, mode='nearest')
    del vol_rec
    gc.collect()

    print(f"[{case_name}] Интерполяция маски (Nearest Neighbor)...")
    vol_seg_iso = zoom(vol_seg, zoom_factors, order=0, mode='nearest')
    del vol_seg
    gc.collect()

    # 3. КОРРЕКТИРОВКА КЛАССОВ (0 - Фон)
    print(f"[{case_name}] Конвертация классов...")
    vol_seg_fixed = np.zeros_like(vol_seg_iso, dtype=np.uint8)
    vol_seg_fixed[vol_seg_iso == 1] = 0  # Фон
    vol_seg_fixed[vol_seg_iso == 2] = 1  # Банка
    vol_seg_fixed[vol_seg_iso == 3] = 2  # Наполнитель
    vol_seg_fixed[vol_seg_iso == 4] = 3  # Твердые предметы
    del vol_seg_iso
    gc.collect()

    # 4. СОХРАНЕНИЕ
    affine = np.eye(4)
    target_spacing = (xy_spacing, xy_spacing, xy_spacing)
    
    print(f"[{case_name}] Экспорт файлов NIfTI...")
    # Снимок
    v_min, v_max = vol_rec_iso.min(), vol_rec_iso.max()
    vol_rec_uint16 = (((vol_rec_iso - v_min) / (v_max - v_min)) * 65535.0).astype(np.uint16)
    nii_rec = nib.Nifti1Image(vol_rec_uint16, affine)
    nii_rec.header.set_zooms(target_spacing)
    nib.save(nii_rec, out_images_dir / f"{case_name}_0000.nii.gz")
    del vol_rec_iso, vol_rec_uint16, nii_rec
    
    # Маска
    nii_lbl = nib.Nifti1Image(vol_seg_fixed, affine)
    nii_lbl.header.set_zooms(target_spacing)
    nib.save(nii_lbl, out_labels_dir / f"{case_name}.nii.gz")
    del vol_seg_fixed, nii_lbl
    gc.collect()
    
    print(f"[{case_name}] ✔ Успешно сохранен!")

def build_full_nnunet_dataset(
    root_dir: str, 
    out_dir: str, 
    dataset_id: int = 101, 
    dataset_name: str = "BankaIndustrial"
):
    """Создает структуру папок, генерирует train/val сеты и пишет dataset.json"""
    # 1. Структура директорий nnU-Net v2
    dataset_folder = Path(out_dir) / f"Dataset{dataset_id}_{dataset_name}"
    imagesTr = dataset_folder / "imagesTr"
    labelsTr = dataset_folder / "labelsTr"
    
    imagesTr.mkdir(parents=True, exist_ok=True)
    labelsTr.mkdir(parents=True, exist_ok=True)

    root_path = Path(root_dir)
    
    # ФИЗИЧЕСКИЕ ПАРАМЕТРЫ ТОМОГРАФА (Настройте под себя)
    XY_SPACING = 0.1138
    Z_SPACING = 1.0

    # ==========================================
    # ГЕНЕРАЦИЯ ДАННЫХ (Можно менять количество срезов)
    # ==========================================
    # Обучающая выборка (Train): Срезы 200 - 264
    process_and_save_chunk(
        root_path=root_path, out_images_dir=imagesTr, out_labels_dir=labelsTr,
        start_slice=200, num_slices=64, case_name="banka_train_01",
        xy_spacing=XY_SPACING, z_spacing=Z_SPACING
    )

    # Валидационная выборка (Val): Срезы 500 - 564 
    # (Берем подальше от Train, чтобы сеть не схитрила, запомнив соседние дефекты)
    process_and_save_chunk(
        root_path=root_path, out_images_dir=imagesTr, out_labels_dir=labelsTr,
        start_slice=500, num_slices=64, case_name="banka_val_01",
        xy_spacing=XY_SPACING, z_spacing=Z_SPACING
    )

    # ==========================================
    # ГЕНЕРАЦИЯ DATASET.JSON
    # ==========================================
    print("\n[*] Генерация dataset.json...")
    dataset_json = {
        "channel_names": {
            # "nonCT" отключает Хаунсфилд-клиппинг и включает Z-score нормализацию!
            "0": "nonCT" 
        },
        "labels": {
            "background": 0,
            "can": 1,
            "filler": 2,
            "hard_subjects": 3
        },
        "numTraining": 2, # У нас 2 объема (train + val)
        "file_ending": ".nii.gz"
    }

    with open(dataset_folder / "dataset.json", "w", encoding="utf-8") as f:
        json.dump(dataset_json, f, indent=4)

    print(f"\n[✔] Датасет полностью готов: {dataset_folder}")
    print("[*] Перенесите эту папку в директорию nnUNet_raw и запустите plan_and_preprocess!")

# ==========================================
# ЗАПУСК
# ==========================================
if __name__ == "__main__":
    # Укажите ВАШИ пути
    root_directory = r"C:\Users\semen\SEprojects\archive\2DeteCT_slices3001-4000_RecSeg"
    output_directory = r"C:\Users\semen\SEprojects\archive\nnUNet_Ready"

    build_full_nnunet_dataset(root_dir=root_directory, out_dir=output_directory)


[banka_train_01] НАЧАЛО ОБРАБОТКИ (Срезы 200 - 264)
[banka_train_01] Загрузка 64 TIFF файлов в ОЗУ...
[banka_train_01] Интерполяция снимка (ось Z увеличится в 8.79 раз)...
[banka_train_01] Интерполяция маски (Nearest Neighbor)...
[banka_train_01] Конвертация классов...
[banka_train_01] Экспорт файлов NIfTI...
[banka_train_01] ✔ Успешно сохранен!

[banka_val_01] НАЧАЛО ОБРАБОТКИ (Срезы 500 - 564)
[banka_val_01] Загрузка 64 TIFF файлов в ОЗУ...
[banka_val_01] Интерполяция снимка (ось Z увеличится в 8.79 раз)...
[banka_val_01] Интерполяция маски (Nearest Neighbor)...
[banka_val_01] Конвертация классов...
[banka_val_01] Экспорт файлов NIfTI...
[banka_val_01] ✔ Успешно сохранен!

[*] Генерация dataset.json...

[✔] Датасет полностью готов: C:\Users\semen\SEprojects\archive\nnUNet_Ready\Dataset101_BankaIndustrial
[*] Перенесите эту папку в директорию nnUNet_raw и запустите plan_and_preprocess!


In [3]:
import sys
print(sys.executable)

c:\Users\semen\AppData\Local\Programs\Python\Python311\python.exe


In [1]:
%pip install nnunetv2

  Using cached argparse-1.4.0-py2.py3-none-any.whl.metadata (2.8 kB)
Using cached argparse-1.4.0-py2.py3-none-any.whl (23 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import numpy as np
import tifffile
import nibabel as nib
import json
import gc
from pathlib import Path
from scipy.ndimage import zoom

def process_and_save_chunk(
    root_path: Path, 
    out_images_dir: Path, 
    out_labels_dir: Path,
    start_slice: int, 
    num_slices: int, 
    case_name: str, 
    xy_spacing: float, 
    z_spacing: float,
    is_tilted: bool = False  # НОВЫЙ ПАРАМЕТР
):
    """Извлекает, интерполирует и сохраняет один кусок данных, экономя ОЗУ."""
    print(f"\n[{case_name}] НАЧАЛО ОБРАБОТКИ (Срезы {start_slice} - {start_slice + num_slices}) | Завален: {is_tilted}")
    
    all_slice_dirs = sorted(root_path.glob("[sS]lice*"))
    actual_end = min(start_slice + num_slices, len(all_slice_dirs))
    target_dirs = all_slice_dirs[start_slice:actual_end]
    
    rec_slices, seg_slices = [], []
    
    # 1. ЧТЕНИЕ ДАННЫХ
    print(f"[{case_name}] Загрузка {len(target_dirs)} TIFF файлов в ОЗУ...")
    for s_dir in target_dirs:
        mode2_dir = s_dir / "mode2"
        if not mode2_dir.exists(): continue
        
        rec_files = list(mode2_dir.glob("*reconstruction*.tif*"))
        seg_files = list(mode2_dir.glob("*segmentation*.tif*"))
        
        if rec_files and seg_files:
            rec_slices.append(tifffile.imread(rec_files[0]))
            seg_slices.append(tifffile.imread(seg_files[0]))

    # (X, Y, Z)
    vol_rec = np.transpose(np.stack(rec_slices), (2, 1, 0))
    vol_seg = np.transpose(np.stack(seg_slices), (2, 1, 0))
    
    del rec_slices, seg_slices
    gc.collect()

    # 2. ИНТЕРПОЛЯЦИЯ ДО КУБА
    z_zoom_factor = z_spacing / xy_spacing
    zoom_factors = (1.0, 1.0, z_zoom_factor)
    
    print(f"[{case_name}] Интерполяция снимка (ось Z увеличится в {z_zoom_factor:.2f} раз)...")
    vol_rec_iso = zoom(vol_rec, zoom_factors, order=3, mode='nearest')
    del vol_rec
    gc.collect()

    print(f"[{case_name}] Интерполяция маски (Nearest Neighbor)...")
    vol_seg_iso = zoom(vol_seg, zoom_factors, order=0, mode='nearest')
    del vol_seg
    gc.collect()

    # 3. КОРРЕКТИРОВКА КЛАССОВ (0 - Фон)
    print(f"[{case_name}] Конвертация классов...")
    vol_seg_fixed = np.zeros_like(vol_seg_iso, dtype=np.uint8)
    vol_seg_fixed[vol_seg_iso == 1] = 0  # Фон
    vol_seg_fixed[vol_seg_iso == 2] = 1  # Банка
    vol_seg_fixed[vol_seg_iso == 3] = 2  # Наполнитель
    vol_seg_fixed[vol_seg_iso == 4] = 3  # Твердые предметы
    del vol_seg_iso
    gc.collect()

    # НОВЫЙ БЛОК: ФИЗИЧЕСКИЙ ПОВОРОТ
    if is_tilted:
        print(f"[{case_name}] Заваливаем банку на бок (поворот 90 градусов по осям X и Z)...")
        # Поворачиваем массивы. Оси (0, 2) меняют местами X и Z.
        vol_rec_iso = np.rot90(vol_rec_iso, k=1, axes=(0, 2))
        vol_seg_fixed = np.rot90(vol_seg_fixed, k=1, axes=(0, 2))

    # 4. СОХРАНЕНИЕ
    affine = np.eye(4)
    target_spacing = (xy_spacing, xy_spacing, xy_spacing)
    
    print(f"[{case_name}] Экспорт файлов NIfTI...")
    # Снимок
    v_min, v_max = vol_rec_iso.min(), vol_rec_iso.max()
    vol_rec_uint16 = (((vol_rec_iso - v_min) / (v_max - v_min)) * 65535.0).astype(np.uint16)
    nii_rec = nib.Nifti1Image(vol_rec_uint16, affine)
    nii_rec.header.set_zooms(target_spacing)
    nib.save(nii_rec, out_images_dir / f"{case_name}_0000.nii.gz")
    del vol_rec_iso, vol_rec_uint16, nii_rec
    
    # Маска
    nii_lbl = nib.Nifti1Image(vol_seg_fixed, affine)
    nii_lbl.header.set_zooms(target_spacing)
    nib.save(nii_lbl, out_labels_dir / f"{case_name}.nii.gz")
    del vol_seg_fixed, nii_lbl
    gc.collect()
    
    print(f"[{case_name}] ✔ Успешно сохранен!")

def build_full_nnunet_dataset(
    root_dir: str, 
    out_dir: str, 
    dataset_id: int = 103,  # Сменил ID, чтобы не затереть ваши прошлые эксперименты
    dataset_name: str = "BankaIndustrial_Tilted"
):
    """Создает структуру папок, генерирует train/val сеты и пишет dataset.json"""
    dataset_folder = Path(out_dir) / f"Dataset{dataset_id}_{dataset_name}"
    imagesTr = dataset_folder / "imagesTr"
    labelsTr = dataset_folder / "labelsTr"
    
    imagesTr.mkdir(parents=True, exist_ok=True)
    labelsTr.mkdir(parents=True, exist_ok=True)

    root_path = Path(root_dir)
    
    XY_SPACING = 0.1138
    Z_SPACING = 1.0

    # ==========================================
    # 1. Обучающая выборка (Train)
    # ==========================================
    process_and_save_chunk(
        root_path=root_path, out_images_dir=imagesTr, out_labels_dir=labelsTr,
        start_slice=200, num_slices=64, case_name="banka_train_01",
        xy_spacing=XY_SPACING, z_spacing=Z_SPACING,
        is_tilted=False
    )

    # ==========================================
    # 2. Валидация: Ровная банка
    # ==========================================
    process_and_save_chunk(
        root_path=root_path, out_images_dir=imagesTr, out_labels_dir=labelsTr,
        start_slice=500, num_slices=64, case_name="banka_val_normal",
        xy_spacing=XY_SPACING, z_spacing=Z_SPACING,
        is_tilted=False
    )

    # ==========================================
    # 3. Валидация: Заваленная банка (Tilted)
    # ==========================================
    process_and_save_chunk(
        root_path=root_path, out_images_dir=imagesTr, out_labels_dir=labelsTr,
        start_slice=700, num_slices=64, case_name="banka_val_tilted",
        xy_spacing=XY_SPACING, z_spacing=Z_SPACING,
        is_tilted=True  # Активируем поворот
    )

    # ==========================================
    # ГЕНЕРАЦИЯ DATASET.JSON
    # ==========================================
    print("\n[*] Генерация dataset.json...")
    dataset_json = {
        "channel_names": {
            "0": "nonCT" 
        },
        "labels": {
            "background": 0,
            "can": 1,
            "filler": 2,
            "hard_subjects": 3
        },
        "numTraining": 3, # Теперь 3 объема
        "file_ending": ".nii.gz"
    }

    with open(dataset_folder / "dataset.json", "w", encoding="utf-8") as f:
        json.dump(dataset_json, f, indent=4)

    print(f"\n[✔] Датасет полностью готов: {dataset_folder}")

if __name__ == "__main__":
    root_directory = r"C:\Users\semen\SEprojects\archive\2DeteCT_slices3001-4000_RecSeg"
    output_directory = r"C:\Users\semen\SEprojects\archive\nnUNet_Ready"

    build_full_nnunet_dataset(root_dir=root_directory, out_dir=output_directory)


[banka_train_01] НАЧАЛО ОБРАБОТКИ (Срезы 200 - 264) | Завален: False
[banka_train_01] Загрузка 64 TIFF файлов в ОЗУ...
[banka_train_01] Интерполяция снимка (ось Z увеличится в 8.79 раз)...
[banka_train_01] Интерполяция маски (Nearest Neighbor)...
[banka_train_01] Конвертация классов...
[banka_train_01] Экспорт файлов NIfTI...
[banka_train_01] ✔ Успешно сохранен!

[banka_val_normal] НАЧАЛО ОБРАБОТКИ (Срезы 500 - 564) | Завален: False
[banka_val_normal] Загрузка 64 TIFF файлов в ОЗУ...
[banka_val_normal] Интерполяция снимка (ось Z увеличится в 8.79 раз)...
[banka_val_normal] Интерполяция маски (Nearest Neighbor)...
[banka_val_normal] Конвертация классов...
[banka_val_normal] Экспорт файлов NIfTI...
[banka_val_normal] ✔ Успешно сохранен!

[banka_val_tilted] НАЧАЛО ОБРАБОТКИ (Срезы 700 - 764) | Завален: True
[banka_val_tilted] Загрузка 64 TIFF файлов в ОЗУ...
[banka_val_tilted] Интерполяция снимка (ось Z увеличится в 8.79 раз)...
[banka_val_tilted] Интерполяция маски (Nearest Neighbor)...

In [1]:
import numpy as np
import tifffile
import nibabel as nib
import gc
from pathlib import Path
from scipy.ndimage import zoom

def process_and_save_chunk(
    root_path: Path, 
    out_images_dir: Path, 
    out_labels_dir: Path,
    start_slice: int, 
    num_slices: int, 
    case_name: str, 
    xy_spacing: float, 
    z_spacing: float,
    is_tilted: bool = False
):
    """Извлекает, интерполирует и сохраняет один кусок данных, экономя ОЗУ."""
    print(f"\n[{case_name}] НАЧАЛО ОБРАБОТКИ (Срезы {start_slice} - {start_slice + num_slices}) | Завален: {is_tilted}")
    
    all_slice_dirs = sorted(root_path.glob("[sS]lice*"))
    actual_end = min(start_slice + num_slices, len(all_slice_dirs))
    target_dirs = all_slice_dirs[start_slice:actual_end]
    
    rec_slices, seg_slices = [], []
    
    print(f"[{case_name}] Загрузка {len(target_dirs)} TIFF файлов в ОЗУ...")
    for s_dir in target_dirs:
        mode2_dir = s_dir / "mode2"
        if not mode2_dir.exists(): continue
        
        rec_files = list(mode2_dir.glob("*reconstruction*.tif*"))
        seg_files = list(mode2_dir.glob("*segmentation*.tif*"))
        
        if rec_files and seg_files:
            rec_slices.append(tifffile.imread(rec_files[0]))
            seg_slices.append(tifffile.imread(seg_files[0]))

    vol_rec = np.transpose(np.stack(rec_slices), (2, 1, 0))
    vol_seg = np.transpose(np.stack(seg_slices), (2, 1, 0))
    
    del rec_slices, seg_slices
    gc.collect()

    z_zoom_factor = z_spacing / xy_spacing
    zoom_factors = (1.0, 1.0, z_zoom_factor)
    
    print(f"[{case_name}] Интерполяция снимка (ось Z увеличится в {z_zoom_factor:.2f} раз)...")
    vol_rec_iso = zoom(vol_rec, zoom_factors, order=3, mode='nearest')
    del vol_rec
    gc.collect()

    print(f"[{case_name}] Интерполяция маски (Nearest Neighbor)...")
    vol_seg_iso = zoom(vol_seg, zoom_factors, order=0, mode='nearest')
    del vol_seg
    gc.collect()

    print(f"[{case_name}] Конвертация классов...")
    vol_seg_fixed = np.zeros_like(vol_seg_iso, dtype=np.uint8)
    vol_seg_fixed[vol_seg_iso == 1] = 0  
    vol_seg_fixed[vol_seg_iso == 2] = 1  
    vol_seg_fixed[vol_seg_iso == 3] = 2  
    vol_seg_fixed[vol_seg_iso == 4] = 3  
    del vol_seg_iso
    gc.collect()

    if is_tilted:
        print(f"[{case_name}] Заваливаем банку на бок (поворот 90 градусов по осям X и Z)...")
        vol_rec_iso = np.rot90(vol_rec_iso, k=1, axes=(0, 2))
        vol_seg_fixed = np.rot90(vol_seg_fixed, k=1, axes=(0, 2))

    affine = np.eye(4)
    target_spacing = (xy_spacing, xy_spacing, xy_spacing)
    
    print(f"[{case_name}] Экспорт файлов NIfTI...")
    v_min, v_max = vol_rec_iso.min(), vol_rec_iso.max()
    vol_rec_uint16 = (((vol_rec_iso - v_min) / (v_max - v_min)) * 65535.0).astype(np.uint16)
    nii_rec = nib.Nifti1Image(vol_rec_uint16, affine)
    nii_rec.header.set_zooms(target_spacing)
    nib.save(nii_rec, out_images_dir / f"{case_name}_0000.nii.gz")
    del vol_rec_iso, vol_rec_uint16, nii_rec
    
    nii_lbl = nib.Nifti1Image(vol_seg_fixed, affine)
    nii_lbl.header.set_zooms(target_spacing)
    nib.save(nii_lbl, out_labels_dir / f"{case_name}.nii.gz")
    del vol_seg_fixed, nii_lbl
    gc.collect()
    
    print(f"[{case_name}] ✔ Успешно сохранен!")

def generate_test_set_only(
    root_dir: str, 
    out_dir: str, 
    dataset_id: int = 103, 
    dataset_name: str = "BankaIndustrial_Tilted"
):
    dataset_folder = Path(out_dir) / f"Dataset{dataset_id}_{dataset_name}"
    
    # Создаем только директории для тестирования
    imagesTs = dataset_folder / "imagesTs"
    labelsTs = dataset_folder / "labelsTs"
    imagesTs.mkdir(parents=True, exist_ok=True)
    labelsTs.mkdir(parents=True, exist_ok=True)

    root_path = Path(root_dir)
    
    XY_SPACING = 0.1138
    Z_SPACING = 1.0

    # 4. ТЕСТОВАЯ ВЫБОРКА: Базовый куб (800-900 срезы)
    process_and_save_chunk(
        root_path=root_path, out_images_dir=imagesTs, out_labels_dir=labelsTs,
        start_slice=800, num_slices=30, case_name="banka_test_base",
        xy_spacing=XY_SPACING, z_spacing=Z_SPACING,
        is_tilted=False
    )
    print(f"\n[✔] Тестовая выборка сохранена в: {imagesTs}")

if __name__ == "__main__":
    root_directory = r"C:\Users\semen\SEprojects\archive\2DeteCT_slices3001-4000_RecSeg"
    output_directory = r"C:\Users\semen\SEprojects\archive\nnUNet_Ready"

    generate_test_set_only(root_dir=root_directory, out_dir=output_directory)


[banka_test_base] НАЧАЛО ОБРАБОТКИ (Срезы 800 - 830) | Завален: False
[banka_test_base] Загрузка 30 TIFF файлов в ОЗУ...
[banka_test_base] Интерполяция снимка (ось Z увеличится в 8.79 раз)...
[banka_test_base] Интерполяция маски (Nearest Neighbor)...
[banka_test_base] Конвертация классов...
[banka_test_base] Экспорт файлов NIfTI...
[banka_test_base] ✔ Успешно сохранен!

[✔] Тестовая выборка сохранена в: C:\Users\semen\SEprojects\archive\nnUNet_Ready\Dataset103_BankaIndustrial_Tilted\imagesTs


In [1]:
import numpy as np
import nibabel as nib
from scipy.ndimage import affine_transform
from scipy.spatial.transform import Rotation as R
from pathlib import Path
import tqdm
import gc

def create_so3_validation_set_memory_safe(
    base_img_path: str, 
    base_lbl_path: str, 
    out_dir: str, 
    n_rotations: int = 20
):
    out_img_dir = Path(out_dir) / "imagesTs_SO3"
    out_lbl_dir = Path(out_dir) / "labelsTs_SO3"
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    print("Загрузка снимка строго в uint16 (~1.8 ГБ)...")
    img_nii = nib.load(base_img_path)
    lbl_nii = nib.load(base_lbl_path)
    
    # ПРЯМОЙ ОБХОД FLOAT64
    img_data = np.asanyarray(img_nii.dataobj, dtype=np.uint16)
    lbl_data = np.asanyarray(lbl_nii.dataobj, dtype=np.uint8)
    
    center = np.array(img_data.shape) / 2.0
    rotations = R.random(n_rotations)
    
    print(f"Последовательная генерация {n_rotations} вращений...")
    
    for i, rot in enumerate(tqdm.tqdm(rotations)):
        inv_matrix = np.linalg.inv(rot.as_matrix())
        offset = center - np.dot(inv_matrix, center)

        # Обработка снимка (order=1 - трилинейная интерполяция)
        rotated_img = affine_transform(img_data, inv_matrix, offset=offset, order=1, mode='constant', cval=0)
        nii_img_rot = nib.Nifti1Image(rotated_img.astype(np.uint16), img_nii.affine, img_nii.header)
        nib.save(nii_img_rot, out_img_dir / f"banka_rot_{i:03d}_0000.nii.gz")
        
        # Очистка памяти от тяжелой копии снимка
        del rotated_img, nii_img_rot
        gc.collect()

        # Обработка маски (order=0 - ближайший сосед)
        rotated_lbl = affine_transform(lbl_data, inv_matrix, offset=offset, order=0, mode='constant', cval=0)
        nii_lbl_rot = nib.Nifti1Image(rotated_lbl, lbl_nii.affine, lbl_nii.header)
        nib.save(nii_lbl_rot, out_lbl_dir / f"banka_rot_{i:03d}.nii.gz")
        
        # Очистка памяти от копии маски
        del rotated_lbl, nii_lbl_rot
        gc.collect()

if __name__ == "__main__":
    base_image = r"C:\Users\semen\SEprojects\archive\nnUNet_Ready\Dataset103_BankaIndustrial_Tilted\imagesTs\banka_test_base_0000.nii.gz"
    base_label = r"C:\Users\semen\SEprojects\archive\nnUNet_Ready\Dataset103_BankaIndustrial_Tilted\labelsTs\banka_test_base.nii.gz"
    output_dataset_dir = r"C:\Users\semen\SEprojects\archive\nnUNet_Ready\Dataset103_BankaIndustrial_Tilted"

    create_so3_validation_set_memory_safe(
        base_img_path=base_image,
        base_lbl_path=base_label,
        out_dir=output_dataset_dir,
        n_rotations=10
    )

Загрузка снимка строго в uint16 (~1.8 ГБ)...
Последовательная генерация 10 вращений...


100%|██████████| 10/10 [07:06<00:00, 42.69s/it]


In [ ]:
import os

# Указываем пути (r-строки спасают от проблем со слешами в Windows)
os.environ['nnUNet_raw'] = r"C:\Users\semen\SEprojects\archive\nnUNet_raw"
os.environ['nnUNet_preprocessed'] = r"C:\Users\semen\SEprojects\archive\nnUNet_preprocessed"
os.environ['nnUNet_results'] = r"C:\Users\semen\SEprojects\archive\nnUNet_results"

: 